In [1]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

c:\Users\Aaru\OneDrive\Desktop\Informal Multilingual RAG\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = PyPDFDirectoryLoader("./data/") 
docs = loader.load()

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200, 
    chunk_overlap=200,
    separators=["\n\n", "\n", "।", ".", " ", ""]
)
chunks = text_splitter.split_documents(docs)

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={'device': 'cuda'} 
)

In [5]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./mumbai_worker_db"
)

In [6]:
batch_size = 50 
for i in range(0, len(chunks), batch_size):
    batch = chunks[i : i + batch_size]
    vector_db.add_documents(batch)